In [19]:
import pandas as pd

# load the data
df = pd.read_csv("../heart.csv")

# perform X/y -split
# if you  have more than one independent variable, list them all here
# leave out the target variable! (dependent variable)

# some missing values here, drop them for SelectKBest
df = df.dropna()



In [20]:
# this just converts the value of column to 0 or 1
# factorize in pandas works too, but only one column at a time
from sklearn.preprocessing import LabelEncoder
variables = ['ExerciseAngina']
encoder = LabelEncoder()
df[variables] = df[variables].apply(encoder.fit_transform)

In [21]:
cut_mapper = {'Down': 1, 'Flat': 2,'Up': 3}
df['ST_Slope'] = df['ST_Slope'].map(cut_mapper)

In [22]:
# this makes multiple columns with the variable (Separate for yes/no)
from sklearn.preprocessing import OneHotEncoder
variables = ['Sex', 'ChestPainType', 'RestingECG']

# use encoder
encoder = OneHotEncoder(sparse_output=False).set_output(transform="pandas")
one_hot_encoded = encoder.fit_transform(df[variables]).astype(int)
df = pd.concat([df,one_hot_encoded],axis=1).drop(columns=variables)

In [23]:
df = df.drop(["Sex_F", "ChestPainType_TA", "RestingECG_LVH"], axis=1)

In [24]:
# this is a nice and common trick => everything EXCEPT target variable => support variable
X = df.drop("HeartDisease", axis=1)

# have only the target variable here (dependent variable)
y = df["HeartDisease"]

In [25]:
from sklearn.feature_selection import SelectKBest
from sklearn.feature_selection import chi2

# convert all continuous variables to integer,
# and convert all negative numbers to 0
X_cat = X.astype(int)
X_cat = X_cat.clip(lower=0)

# initialize chi2 and SelectKBest
# Note: chi2 -test is a very common test
# in statistics and quantitative analysis
# basically it studies the data whether variables are related
# or independent of each other
chi_2_features = SelectKBest(chi2, k=len(X_cat.columns))

# fit our data to the SelectKBest
best_features = chi_2_features.fit(X_cat,y.astype(int))

# use decimal format in table print later
pd.options.display.float_format = '{:.2f}'.format

# wrap it up, and show the results
# the higher the score, the more effect that column has on target variable
df_features = pd.DataFrame(best_features.scores_)
df_columns = pd.DataFrame(X_cat.columns)
f_scores = pd.concat([df_columns,df_features],axis=1)
f_scores.columns = ['Features','Score']
f_scores.sort_values(by='Score',ascending=False)

,Features,Score
2,Cholesterol,2989.56
4,MaxHR,696.65
6,Oldpeak,195.50
5,ExerciseAngina,133.64
0,Age,121.29
10,ChestPainType_ATA,120.35
9,ChestPainType_ASY,112.67
3,FastingBS,50.30
7,ST_Slope,44.68
11,ChestPainType_NAP,32.43
